# AlphaZero Gomoku Training on Google Colab

This notebook trains an AlphaZero agent to play Gomoku (9x9, 5-in-a-row) using GPU acceleration.

**Before starting:**
1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → GPU (T4)
2. **Upload your code**: Upload `alphazero-gomoku.zip` to Google Drive (or use GitHub)

**Expected performance:**
- ~5-15 minutes per iteration on Colab GPU (T4)
- ~5-10x faster than CPU training

## 1. Setup Environment

In [ ]:
# Verify GPU is enabled
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Go to Runtime → Change runtime type → GPU")

## 2. Load Code

### Option A: From Google Drive (Recommended)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Extract code from zip file
# First, upload 'alphazero-gomoku.zip' to your Google Drive root
!unzip -q /content/drive/MyDrive/alphazero-gomoku.zip -d /content/
%cd /content/Reinforcement-learning-CW
!ls -la

### Option B: From GitHub

In [ ]:
# Clone from GitHub (if you have pushed your code)
# !git clone https://github.com/yourusername/your-repo.git
# %cd your-repo

## 3. Install Dependencies

In [ ]:
# Install required packages
!pip install -q pyyaml

# Note: torch and numpy are pre-installed in Colab
print("✓ Dependencies installed")

## 4. Configure Training (Optional)

Adjust hyperparameters if needed. Default settings work well for Colab GPU.

In [ ]:
# View current configuration
!cat config.yaml

In [ ]:
# Optional: Modify config for faster testing
# Uncomment and run if you want quicker iterations for testing

# import yaml
# with open('config.yaml', 'r') as f:
#     config = yaml.safe_load(f)
# 
# # Faster settings for testing
# config['num_simulations'] = 200  # Faster MCTS
# config['games_per_iteration'] = 50  # Fewer games
# config['checkpoint_freq'] = 10  # Save more often
# 
# with open('config.yaml', 'w') as f:
#     yaml.dump(config, f)
# 
# print("✓ Config modified for faster training")

## 5. Train AlphaZero Agent

Start training! Monitor the output for:
- `Device: cuda` ✓ (confirms GPU usage)
- Loss values (should decrease over time)
- Iteration progress

In [ ]:
# Fresh training from scratch
# Start with 50 iterations (~4-12 hours on T4 GPU)
!python scripts/train.py --config config.yaml --iterations 50

In [ ]:
# Resume training from checkpoint (if session disconnects)
# !python scripts/train.py --resume checkpoints/checkpoint_10.pt --iterations 50

## 6. Evaluate Trained Agent

In [ ]:
# Evaluate against random baseline
!python scripts/evaluate.py \
    --agent1 checkpoints/checkpoint_50.pt \
    --agent2 random \
    --games 20 \
    --simulations 400

In [ ]:
# Compare two checkpoints to see improvement
# !python scripts/evaluate.py \
#     --agent1 checkpoints/checkpoint_50.pt \
#     --agent2 checkpoints/checkpoint_10.pt \
#     --games 20

## 7. Monitor Training Progress

In [ ]:
# List all saved checkpoints
!ls -lh checkpoints/

In [ ]:
# Check GPU memory usage during training
import torch
if torch.cuda.is_available():
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"GPU Memory Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

## 8. Save Checkpoints to Google Drive

**Important:** Colab sessions are temporary! Save your trained models to Drive.

In [ ]:
# Copy all checkpoints to Google Drive for persistence
!mkdir -p /content/drive/MyDrive/alphazero-checkpoints
!cp -r checkpoints/* /content/drive/MyDrive/alphazero-checkpoints/
!ls -lh /content/drive/MyDrive/alphazero-checkpoints/

print("\n✓ Checkpoints backed up to Google Drive!")

## 9. Download Checkpoints (Alternative)

Download specific checkpoints directly to your computer.

In [ ]:
# Download best checkpoint
from google.colab import files

# Download latest checkpoint
import os
checkpoints = sorted([f for f in os.listdir('checkpoints') if f.endswith('.pt')])
if checkpoints:
    latest = checkpoints[-1]
    print(f"Downloading {latest}...")
    files.download(f'checkpoints/{latest}')
else:
    print("No checkpoints found yet!")

## Tips for Long Training Sessions

1. **Save frequently**: Checkpoints are saved every 10 iterations by default
2. **Use Colab Pro**: For longer sessions (up to 24 hours) and better GPUs
3. **Monitor usage**: Free Colab has usage limits; Pro removes them
4. **Resume training**: If disconnected, restore from last checkpoint

## Expected Results

- **10 iterations**: Beats random 60-70%
- **50 iterations**: Beats random >90%
- **100 iterations**: Strong intermediate player
- **500+ iterations**: Expert level

## Troubleshooting

**"CUDA out of memory"**:
```python
# Reduce batch size in config.yaml
batch_size: 128  # instead of 256
```

**Session disconnected**:
```python
# Resume from last checkpoint
!python scripts/train.py --resume checkpoints/checkpoint_XX.pt
```

**Need faster iterations**:
```python
# Reduce simulations or games in config.yaml
num_simulations: 200
games_per_iteration: 50
```